# 🐙🦄🐉🐕🦍🫧 KIN — Train All 6 Companions on Gemma 4 31B

**Model:** Gemma 4 31B (31 billion parameters)
**GPU:** A100 40GB (Colab Pro+)
**Pipeline:** SFT → SimPO → GRPO → GGUF Export
**Budget:** ~720 compute units (~$100 total)
**Timeline:** 5 days, run one notebook session per day

| Day | Companions | Est. Units |
|-----|-----------|------------|
| 1 | Cipher (SFT+SimPO) | ~83 |
| 2 | Cipher (GRPO+Export) + Forge (SFT+SimPO) | ~136 |
| 3 | Forge (GRPO) + Vortex (full) | ~175 |
| 4 | Mischief + Aether (full) | ~205 |
| 5 | Catalyst (full) + Eval + Push | ~121 |

---

In [ ]:
# ═══════════════════════════════════════════════════
# SETUP (Run once per session)
# ═══════════════════════════════════════════════════
!pip install unsloth[colab-new] -q
!pip install trl datasets -q

import torch
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'GPU: {gpu} | VRAM: {vram:.1f} GB')

if vram < 35:
    print('⚠️ Need A100 for 31B. Switch to E4B or upgrade to Colab Pro+')
    MODEL = 'unsloth/gemma-4-E4B-it-bnb-4bit'
    LORA_R, LORA_A, SEQ_LEN = 32, 64, 4096
    print(f'Falling back to E4B')
else:
    MODEL = 'unsloth/gemma-4-31B-it-bnb-4bit'
    LORA_R, LORA_A, SEQ_LEN = 64, 128, 8192
    print(f'✅ Using 31B — maximum quality')

In [ ]:
# ═══════════════════════════════════════════════════
# CONFIGURATION — Set which companion to train
# ═══════════════════════════════════════════════════

# 🔧 CHANGE THIS each day:
COMPANION = 'cipher'  # cipher | forge | vortex | mischief | aether | catalyst
STAGES = ['sft', 'simpo', 'grpo']  # Which stages to run this session
PUSH_TO_HUB = False  # Set True on Day 5 to upload
HF_REPO = f'kr8tiv/kin-{COMPANION}-GGUF'

# Companion metadata
COMPANIONS = {
    'cipher':   {'emoji': '🐙', 'name': 'Code Kraken',   'system': 'You are Cipher, the Code Kraken — a design-obsessed AI companion who builds exceptional websites while teaching about design, development, and creative technology.'},
    'forge':    {'emoji': '🦄', 'name': 'Cyber Unicorn',  'system': 'You are Forge, the Cyber Unicorn — a patient, precise programming companion who helps developers build anything. Part mentor, part pair-programming partner.'},
    'vortex':   {'emoji': '🐉', 'name': 'Teal Dragon',    'system': 'You are Vortex, the Teal Dragon — a tireless marketing companion who breathes creative fire into brand strategy. Always strategizing, always optimizing.'},
    'mischief': {'emoji': '🐕', 'name': 'Glitch Pup',     'system': 'You are Mischief, the Glitch Pup — a playful, protective family companion and personal brand whisperer. Part family pet, part brand strategist.'},
    'aether':   {'emoji': '🦍', 'name': 'Frost Ape',      'system': 'You are Aether, the Frost Ape — a wise creative companion who dwells in the mountains of imagination. Part muse, part editor.'},
    'catalyst': {'emoji': '🫧', 'name': 'Cosmic Blob',    'system': 'You are Catalyst, the Cosmic Blob — a transformation companion who helps you become the person you want to be. Part life coach, part financial advisor.'},
}

info = COMPANIONS[COMPANION]
print(f"{info['emoji']} Training: {COMPANION.upper()} — {info['name']}")
print(f"Stages: {' → '.join(s.upper() for s in STAGES)}")
print(f"Model: {MODEL}")

In [ ]:
# ═══════════════════════════════════════════════════
# LOAD MODEL
# ═══════════════════════════════════════════════════
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_A,
    lora_dropout=0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
)

print(f'{info["emoji"]} Model loaded with LoRA r={LORA_R}')

In [ ]:
# ═══════════════════════════════════════════════════
# GENERATE TRAINING DATA (templates — zero cost)
# ═══════════════════════════════════════════════════
import json, random
from datasets import Dataset
from pathlib import Path

sys_prompt = info['system']

# Generate SFT examples
sft_examples = []
topics = ['getting started', 'best practices', 'common mistakes',
          'advanced techniques', 'tools and workflows', 'case studies',
          'troubleshooting', 'optimization', 'collaboration', 'planning']

for i in range(2000):
    topic = random.choice(topics)
    text = f'<start_of_turn>system\n{sys_prompt}<end_of_turn>\n'
    text += f'<start_of_turn>user\nHelp me with {topic} for my project.<end_of_turn>\n'
    text += f'<start_of_turn>model\nGreat question! Let me help you with {topic}. '
    text += f'Here\'s my approach as {info["name"]}...\n\n'
    text += f'**Step 1:** Understand your current situation\n'
    text += f'**Step 2:** Define clear objectives\n'
    text += f'**Step 3:** Create an actionable plan\n'
    text += f'**Step 4:** Execute and iterate\n\n'
    text += f'The key is consistency and quality. Want me to dive deeper?<end_of_turn>\n'
    sft_examples.append({'text': text})

sft_dataset = Dataset.from_list(sft_examples)
print(f'{info["emoji"]} Generated {len(sft_dataset)} SFT examples')

# Generate preference pairs for SimPO
pref_pairs = []
for i in range(500):
    topic = random.choice(topics)
    pref_pairs.append({
        'prompt': f'Help me with {topic}',
        'chosen': f'As {info["name"]}, I\'d love to help with {topic}! Let me share a structured approach with clear steps and expert insights...',
        'rejected': f'Sure. Just do {topic}. It\'s not that hard.'
    })
pref_dataset = Dataset.from_list(pref_pairs)
print(f'{info["emoji"]} Generated {len(pref_dataset)} preference pairs')

In [ ]:
# ═══════════════════════════════════════════════════
# STAGE 1: SFT
# ═══════════════════════════════════════════════════
if 'sft' in STAGES:
    from trl import SFTTrainer
    from transformers import TrainingArguments

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=sft_dataset,
        args=TrainingArguments(
            output_dir=f'/tmp/kin-{COMPANION}-sft',
            num_train_epochs=2,
            per_device_train_batch_size=1 if '31B' in MODEL else 2,
            gradient_accumulation_steps=16 if '31B' in MODEL else 8,
            learning_rate=2e-4,
            lr_scheduler_type='cosine',
            warmup_ratio=0.03,
            bf16=True,
            optim='adamw_8bit',
            logging_steps=25,
            save_steps=200,
            seed=42,
            report_to='none',
        ),
        dataset_text_field='text',
        max_seq_length=SEQ_LEN,
        packing=True,
    )

    print(f'{info["emoji"]} Starting SFT...')
    trainer.train()
    print(f'{info["emoji"]} ✅ SFT complete!')
    model.save_pretrained(f'/tmp/kin-{COMPANION}-sft/lora')
    tokenizer.save_pretrained(f'/tmp/kin-{COMPANION}-sft/lora')

In [ ]:
# ═══════════════════════════════════════════════════
# STAGE 2: SimPO
# ═══════════════════════════════════════════════════
if 'simpo' in STAGES:
    from trl import CPOTrainer, CPOConfig

    cpo_trainer = CPOTrainer(
        model=model,
        args=CPOConfig(
            output_dir=f'/tmp/kin-{COMPANION}-simpo',
            num_train_epochs=1,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            learning_rate=5e-7,
            loss_type='simpo',
            bf16=True,
            logging_steps=10,
            save_steps=100,
            report_to='none',
        ),
        train_dataset=pref_dataset,
        tokenizer=tokenizer,
    )

    print(f'{info["emoji"]} Starting SimPO...')
    cpo_trainer.train()
    print(f'{info["emoji"]} ✅ SimPO complete!')
    model.save_pretrained(f'/tmp/kin-{COMPANION}-simpo/lora')
    tokenizer.save_pretrained(f'/tmp/kin-{COMPANION}-simpo/lora')

In [ ]:
# ═══════════════════════════════════════════════════
# STAGE 3: GRPO (Text-based rewards — no render loop needed)
# ═══════════════════════════════════════════════════
if 'grpo' in STAGES:
    from trl import GRPOTrainer, GRPOConfig

    # Simple reward: personality adherence + quality
    def companion_reward(completions, **kwargs):
        rewards = []
        name_lower = info['name'].lower()
        for c in completions:
            text = c[0]['content'] if isinstance(c, list) else c
            score = 0.3  # base
            if len(text) > 100: score += 0.1  # substantive
            if '**' in text: score += 0.1  # formatting
            if any(w in text.lower() for w in ['step', 'first', 'approach']): score += 0.15  # structured
            if len(text) < 2000: score += 0.1  # concise (SimPO-like)
            if any(w in text.lower() for w in ['here\'s', 'let me', 'great question']): score += 0.15  # engaged
            rewards.append(min(1.0, score))
        return rewards

    grpo_prompts = Dataset.from_list([
        {'prompt': [{'role': 'user', 'content': f'As {info["name"]}, help me with: {random.choice(topics)}'}]}
        for _ in range(300)
    ])

    grpo_trainer = GRPOTrainer(
        model=model,
        args=GRPOConfig(
            output_dir=f'/tmp/kin-{COMPANION}-grpo',
            num_train_epochs=1,
            per_device_train_batch_size=2,
            learning_rate=8e-6,
            num_generations=2,
            max_prompt_length=1024,
            max_completion_length=2048,
            bf16=True,
            logging_steps=5,
            report_to='none',
        ),
        train_dataset=grpo_prompts,
        tokenizer=tokenizer,
        reward_funcs=[companion_reward],
    )

    print(f'{info["emoji"]} Starting GRPO...')
    grpo_trainer.train()
    print(f'{info["emoji"]} ✅ GRPO complete!')

In [ ]:
# ═══════════════════════════════════════════════════
# EXPORT GGUF + PUSH TO HUB
# ═══════════════════════════════════════════════════
print(f'{info["emoji"]} Exporting GGUF...')
model.save_pretrained_gguf(
    f'/tmp/kin-{COMPANION}-gguf',
    tokenizer,
    quantization_method=['q4_k_m', 'q5_k_m'],
)

import os
for f in os.listdir(f'/tmp/kin-{COMPANION}-gguf'):
    size = os.path.getsize(f'/tmp/kin-{COMPANION}-gguf/{f}') / 1e9
    print(f'  {f}: {size:.2f} GB')

if PUSH_TO_HUB:
    print(f'Pushing to {HF_REPO}...')
    model.push_to_hub_gguf(
        HF_REPO,
        tokenizer,
        quantization_method=['q4_k_m', 'q5_k_m'],
        # token='hf_YOUR_TOKEN',  # Uncomment and add your token
    )
    print(f'✅ Pushed to {HF_REPO}')

print(f'\n{info["emoji"]} {COMPANION.upper()} TRAINING COMPLETE!')
print(f'Download GGUF from: /tmp/kin-{COMPANION}-gguf/')
print(f'Then locally: ollama create kin-{COMPANION} -f Modelfile')

## Compute Budget Check

Run this after each session to track spending:

In [ ]:
# Check remaining compute units
# Go to: https://colab.research.google.com/signup to see your balance
print('\n═══════════════════════════════════════════')
print('  Budget Tracker')
print('═══════════════════════════════════════════')
print(f'  Companion trained: {COMPANION}')
print(f'  Stages completed: {STAGES}')
print(f'  Model size: {"31B" if "31B" in MODEL else "E4B"}')
print(f'')
print(f'  Day 1 (Cipher SFT+SimPO):     ~83 units')
print(f'  Day 2 (Cipher GRPO + Forge):   ~136 units')
print(f'  Day 3 (Forge GRPO + Vortex):   ~175 units')
print(f'  Day 4 (Mischief + Aether):     ~205 units')
print(f'  Day 5 (Catalyst + QA + Push):  ~121 units')
print(f'  ─────────────────────────────────────')
print(f'  Total estimate:                ~720 units')
print(f'  Budget (Pro+ + $50 PAYG):      1100 units')
print(f'  Buffer:                        ~380 units (34%)')
print(f'')
print(f'  Check balance: https://colab.research.google.com/signup')